In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\mannd\.cache\kagglehub\datasets\abdallahalidev\plantvillage-dataset\versions\3


use this to know the current dir

In [3]:
# import os
# os.listdir(path+'/plantvillage dataset/color')


In [4]:
train_transforms = transforms.Compose([
    # Flips
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),

    # 2. Combined Rotation and Safe Zoom
    transforms.RandomAffine(degrees=15, scale=(0.95, 1.05)),

    transforms.Resize((224, 224)),
    # 0 ton 1
    transforms.ToTensor(),
    # form -1 to 1 and also then again shift so that it matches wiht the basemodel SA imagenet
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# one for training other ofr validation this one will not have any augmentation
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

before processing

In [11]:
import os

os.listdir(path+'/plantvillage dataset')
raw=path+'/plantvillage dataset/color'
print(os.listdir(raw),len(os.listdir(raw)),sep="\n")

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite',

In [12]:
import os
import shutil

# 1. Define your paths
# Where your current 38 messy folders are sitting
source_dir = raw

# The new master folder "just outside the path" (e.g., going up one level)
# Change dest_dir to a writable location, like /kaggle/working/
dest_dir = '/kaggle/working/processed_plantvillage_dataset'
print(f"the main dir {source_dir}\n the one where we coping the data {dest_dir}")

os.makedirs(dest_dir, exist_ok=True)

# the 38 folders
for folder_name in os.listdir(source_dir):
    old_folder_path = os.path.join(source_dir, folder_name)

    if '___' in folder_name:

        # 'Tomato___Late_blight' -> grabs 'Late_blight'
        disease_name = folder_name.split('___')[1]

        # dest_dir+disease name
        disease_folder_path = os.path.join(dest_dir, disease_name)
        os.makedirs(disease_folder_path, exist_ok=True)

        # 5. Go inside the old folder and grab the raw images
        for image_name in os.listdir(old_folder_path):
            old_image_path = os.path.join(old_folder_path, image_name)
            # This guarantees that 'image_01.jpg' from Tomato doesn't overwrite 'image_01.jpg' from Potato.
            new_image_name = f"{folder_name}_{image_name}"
            new_image_path = os.path.join(disease_folder_path, new_image_name)

            # Move the image directly into the master disease folder
            # We need to copy the files since the source is in a read-only location.
            # shutil.move(old_image_path, new_image_path)
            shutil.copy(old_image_path, new_image_path)

        print(f"Extracted all images from {folder_name} into {disease_name} ")

print("\nDataset successfully re-routed purely by disease feature!")

the main dir C:\Users\mannd\.cache\kagglehub\datasets\abdallahalidev\plantvillage-dataset\versions\3/plantvillage dataset/color
 the one where we coping the data /kaggle/working/processed_plantvillage_dataset
Extracted all images from Apple___Apple_scab into Apple_scab 
Extracted all images from Apple___Black_rot into Black_rot 
Extracted all images from Apple___Cedar_apple_rust into Cedar_apple_rust 
Extracted all images from Apple___healthy into healthy 
Extracted all images from Blueberry___healthy into healthy 
Extracted all images from Cherry_(including_sour)___healthy into healthy 
Extracted all images from Cherry_(including_sour)___Powdery_mildew into Powdery_mildew 
Extracted all images from Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot into Cercospora_leaf_spot Gray_leaf_spot 
Extracted all images from Corn_(maize)___Common_rust_ into Common_rust_ 
Extracted all images from Corn_(maize)___healthy into healthy 
Extracted all images from Corn_(maize)___Northern_Leaf_Blight 

some more leaks

In [1]:
rename_map = {
    'Tomato_mosaic_virus': 'Mosaic_virus',
    'Tomato_Yellow_Leaf_Curl_Virus': 'Yellow_Leaf_Curl_Virus',
    'Common_rust_': 'Common_rust',
    'Spider_mites Two-spotted_spider_mite': 'Spider_mites'
}
#  'Tomato_mosaic_virus',
#  'Tomato_Yellow_Leaf_Curl_Virus'

for old_name, new_name in rename_map.items():
    old_path = os.path.join(dest_dir, old_name)
    new_path = os.path.join(dest_dir, new_name)

    
    os.rename(old_path, new_path)
    print(f"Renamed: {old_name} -> {new_name}")
    
        # print(f"Skipped: {old_name} (Already renamed or not found)")

print("\nClass names are now perfectly abstracted!")

NameError: name 'os' is not defined

In [17]:
main=dest_dir
os.listdir(dest_dir)

['Apple_scab',
 'Bacterial_spot',
 'Black_rot',
 'Cedar_apple_rust',
 'Cercospora_leaf_spot Gray_leaf_spot',
 'Common_rust',
 'Common_rust_',
 'Early_blight',
 'Esca_(Black_Measles)',
 'Haunglongbing_(Citrus_greening)',
 'healthy',
 'Late_blight',
 'Leaf_blight_(Isariopsis_Leaf_Spot)',
 'Leaf_Mold',
 'Leaf_scorch',
 'Mosaic_virus',
 'Northern_Leaf_Blight',
 'Powdery_mildew',
 'Septoria_leaf_spot',
 'Spider_mites',
 'Spider_mites Two-spotted_spider_mite',
 'Target_Spot',
 'Tomato_mosaic_virus',
 'Tomato_Yellow_Leaf_Curl_Virus',
 'Yellow_Leaf_Curl_Virus']

lets split and load

In [9]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader, Subset


#  Create two identical blueprints of the dataset idk we we can have a same dataset and
# but attach the different Compose pipelines we built earlier
train_dataset = datasets.ImageFolder(root=main, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=main, transform=test_transforms)

# 80/20 split
total_images = len(train_dataset)
train_size = int(0.8 * total_images)
valid_size = total_images - train_size

print(f"total images: {total_images} \t training on: {train_size}  validating on: {valid_size}")

# randperm give random no form 0 to total_images but it retursn tensor and we covert tensor to tolist usint the tolist()
# this is way faster than normal random int.
indices = torch.randperm(total_images).tolist()

# Route the shuffled indices to the correct subsets
# The first 80% get the training transforms (wiht some noise - augmentation)
train_subset = Subset(train_dataset, indices[:train_size])
# The remaining 20% get the validation transforms (resizing only)
valid_subset = Subset(val_dataset, indices[train_size:])

# Finally, load them into the DataLoaders
train_loader = DataLoader(train_subset, batch_size=16 , shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_subset, batch_size=16 , shuffle=False, num_workers=2)

total images: 62903 	 training on: 50322  validating on: 12581


In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

# 1. Hardware Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Allocation: {device}")

# 2. Download the pre-trained MobileNetV2
print("Downloading MobileNetV2...")
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# 3. Freeze the "Eyes" (The Feature Extractor)
# We don't want to destroy the visual patterns it already learned from Google
for param in model.parameters():
    param.requires_grad = False

# 4. Re-engineer the "Brain" (The Classifier Head)
# Dynamically get the number of diseases based on the folders we organized earlier
num_classes = len(train_dataset.classes)

# Replace the final layer to output our specific number of diseases
model.classifier[1] = nn.Linear(in_features=model.last_channel, out_features=num_classes)

# 5. Push the new model to the GPU/CPU
model = model.to(device)

# 6. Setup the Optimizer and Loss Function
# CRITICAL: Notice we only pass `model.classifier.parameters()` to the optimizer!
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print(f"MobileNet ready! Adapted to output {num_classes} plant diseases.")